##### Регрессия для IC50

In [37]:
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
from sklearn.linear_model import Ridge

In [38]:

# Загрузка подготовленных данных
edata = pd.read_csv('edata.csv')
edata

,"IC50, mM","CC50, mM",SI,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,ExactMolWt,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiophene,fr_unbrch_alkane,fr_urea,HydrogenMass
0,6.239374,175.482382,28.125000,5.094096,5.094096,0.387225,0.387225,0.417362,42.928571,384.350449,...,0,0,0,0,0,0,0,3,0,44.352
1,0.771831,5.402819,7.000000,3.961417,3.961417,0.533868,0.533868,0.462473,45.214286,388.381750,...,0,0,0,0,0,0,0,3,0,48.384
2,223.808778,161.142320,0.720000,2.627117,2.627117,0.543231,0.543231,0.260923,42.187500,446.458903,...,0,0,0,0,0,0,0,3,0,58.464
3,107.131532,139.270991,1.300000,5.150510,5.150510,0.270476,0.270476,0.429038,36.514286,466.334799,...,0,0,0,0,0,0,0,0,0,42.336
4,15.037911,30.075821,2.000000,5.758408,5.758408,0.278083,0.278083,0.711012,28.600000,332.225249,...,0,0,0,0,0,0,0,0,0,28.224
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
729,31.000104,34.999650,1.129017,12.934891,12.934891,0.048029,-0.476142,0.382752,49.133333,414.240624,...,0,0,0,0,0,0,0,0,0,34.272
730,31.999934,33.999415,1.062484,13.635345,13.635345,0.030329,-0.699355,0.369425,44.542857,485.277738,...,0,0,0,0,0,0,0,0,0,39.312
731,30.999883,33.999458,1.096761,13.991690,13.991690,0.026535,-0.650790,0.284923,41.973684,545.281109,...,1,0,0,0,0,0,0,0,0,43.344
732,31.998959,32.999644,1.031272,13.830180,13.830180,0.146522,-1.408652,0.381559,39.000000,522.282883,...,0,0,0,0,0,0,0,0,0,42.336


In [39]:

# Разделение признаков и целевой переменной
X = edata.drop(columns=['IC50, mM', 'CC50, mM', 'SI'])
y = edata['IC50, mM']

In [40]:
# Разделим выборку на тестовую и тренировочную
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [41]:
# Функция обучения и оценки модели
def eval_fit_model(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = mean_squared_error(y_test, y_pred)**0.5
    r2 = r2_score(y_test, y_pred)
    return {'MAE': mae, 'RMSE': rmse, 'R2': r2}

# Функция оценки модели
def eval_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = mean_squared_error(y_test, y_pred)**0.5
    r2 = r2_score(y_test, y_pred)
    return {'MAE': mae, 'RMSE': rmse, 'R2': r2}

Будем используем модели CatBoost, XGBoost и RandomForest, поскольку:
у нас относительно небольшая выборка (менее 1000 объектов),
присутствует много признаков, среди которых возможна мультиколлинеарность,
а также есть признаки с выбросами и неоднородными масштабами.
LinearRegression возьмем для сравнения в качестве простого ориентира.


In [42]:
# Список моделей
models = {
    "CatBoost": CatBoostRegressor(verbose=0, random_state=42),
    "XGBoost": XGBRegressor(verbosity=0, random_state=42),
    "RandomForest": RandomForestRegressor(random_state=42),
    "LinearRegression": LinearRegression(),
}


##### Обучим модели с применением базовых настроек

In [43]:
# Обучение и оценка
results = {name: eval_fit_model(model, X_train, X_test, y_train, y_test) for name, model in models.items()}
results_df = pd.DataFrame(results).T
print(results_df)

                         MAE        RMSE        R2
CatBoost          108.914608  168.576913  0.043654
XGBoost           108.215965  173.192053 -0.009427
RandomForest      105.600725  161.608399  0.121085
LinearRegression  118.785638  167.501013  0.055822


Стандартизируем признаки и применим метод главных компонент (PCA) для уменьшения размерности признакового пространства, сохранив при этом 90% дисперсии исходных данных.

In [44]:
# Стандартизация
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# PCA с сохранением 90% дисперсии 
pca = PCA(n_components=0.9)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

# Преобразуем numpy в DataFrame
pca_columns = [f'pca_{i}' for i in range(X_train_pca.shape[1])]
X_train_pca = pd.DataFrame(X_train_pca, columns=pca_columns, index=X_train.index)
X_test_pca = pd.DataFrame(X_test_pca, columns=pca_columns, index=X_test.index)

In [45]:
# Возьмем Ridge вместо LinearRegression
models = {
    "CatBoost": CatBoostRegressor(verbose=0, random_state=42),
    "XGBoost": XGBRegressor(verbosity=0, random_state=42),
    "RandomForest": RandomForestRegressor(random_state=42),
    "LinearRegression": LinearRegression()
}

In [46]:
# Обучение моделей, получение метрик 
results_pca = {
    name: eval_fit_model(model, X_train_pca, X_test_pca, y_train, y_test)
    for name, model in models.items()
}

results_pca_df = pd.DataFrame(results_pca).T
print(results_pca_df)

                         MAE        RMSE        R2
CatBoost          108.693438  165.060619  0.083134
XGBoost           109.748154  166.924359  0.062312
RandomForest      106.222780  154.074893  0.201118
LinearRegression  111.566158  158.850164  0.150831


PCA положительно сказалось на RandomForest и LinearRegression.

Теперь подберем гиперпараметры с помощью GridSearchCV. Вместо LinearRegression будем использовать Ridge, поскольку у LinearRegression отсутствуют гиперпараметры для настройки.

In [47]:
 # GridSearchCV параметры
rf_params = {
    'n_estimators': [100, 150, 200],
    'max_depth': [10, 15, 20],
    'min_samples_split': [2, 4, 6],
    'min_samples_leaf': [1, 3, 5],
}

xgb_params = {
    'n_estimators': [100, 200],
    'max_depth': [4, 6, 8],
    'learning_rate': [0.05, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'reg_alpha': [0, 0.1, 1],
    'reg_lambda': [1, 5],
    'booster': ['gbtree'], 
    'tree_method': ['gpu_hist']
}

ridge_params = {
    'alpha': [0.1, 1, 10, 100]
}

cat_params = {
    'depth': [4, 6, 8],
    'learning_rate': [0.05, 0.1, 0.2],
    'iterations': [100, 300],
    'l2_leaf_reg': [1, 5, 9]
}


# Модели с GridSearch
rf = GridSearchCV(
    RandomForestRegressor(random_state=42),
    rf_params,
    cv=3,
    n_jobs=-1,
    verbose=1,
    scoring='neg_mean_squared_error'
)

xgb = GridSearchCV(
    XGBRegressor(random_state=42, verbosity=0),
    param_grid=xgb_params,
    cv=3,
    scoring='neg_mean_squared_error',
    n_jobs=1,
    verbose=1,
    error_score='raise'
)

ridge = GridSearchCV(
    Ridge(),
    ridge_params,
    cv=3,
    n_jobs=-1,
    verbose=1,
    scoring='neg_mean_squared_error'
)

cat = GridSearchCV(
    CatBoostRegressor(verbose=0, random_state=42),
    param_grid=cat_params,
    cv=3,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=1
)

In [48]:
# Обучение моделей с подбором параметров
rf.fit(X_train_pca, y_train)
xgb.fit(X_train_pca, y_train)
ridge.fit(X_train_pca, y_train)
cat.fit(X_train_pca, y_train)

# Метрики моделей с лучшими параметрами
results = {
    "Best RandomForest": eval_model(rf.best_estimator_, X_test_pca, y_test),
    "Best XGBoost": eval_model(xgb.best_estimator_, X_test_pca, y_test),
    "Best Ridge": eval_model(ridge.best_estimator_, X_test_pca, y_test),
    "Best CatBoost": eval_model(cat.best_estimator_, X_test_pca, y_test),
}

# Посмотрим лучшие параметры
print("Best Params RF:", rf.best_params_)
print("Best Params XGB:", xgb.best_params_)
print("Best Params Ridge:", ridge.best_params_)
print("Best Params Cat:", cat.best_params_)

# Посмотрим метрики
print("\n Metrics")
print(pd.DataFrame(results).T)

Fitting 3 folds for each of 81 candidates, totalling 243 fits
Fitting 3 folds for each of 432 candidates, totalling 1296 fits
Fitting 3 folds for each of 4 candidates, totalling 12 fits
Fitting 3 folds for each of 54 candidates, totalling 162 fits
Best Params RF: {'max_depth': 15, 'min_samples_leaf': 5, 'min_samples_split': 2, 'n_estimators': 200}
Best Params XGB: {'booster': 'gbtree', 'colsample_bytree': 1.0, 'learning_rate': 0.05, 'max_depth': 4, 'n_estimators': 100, 'reg_alpha': 0.1, 'reg_lambda': 5, 'subsample': 0.8, 'tree_method': 'gpu_hist'}
Best Params Ridge: {'alpha': 100}
Best Params Cat: {'depth': 8, 'iterations': 100, 'l2_leaf_reg': 5, 'learning_rate': 0.05}

 Metrics
                          MAE        RMSE        R2
Best RandomForest  104.684306  153.171564  0.210458
Best XGBoost       105.367971  156.056330  0.180438
Best Ridge         111.194712  158.384245  0.155805
Best CatBoost      109.347520  159.329931  0.145694


In [51]:
# Обучение моделей с подбором параметров
rf.fit(X_train, y_train)
xgb.fit(X_train, y_train)
cat.fit(X_train, y_train)

# Метрики моделей с лучшими параметрами
results = {
    "Best RandomForest": eval_model(rf.best_estimator_, X_test, y_test),
    "Best XGBoost": eval_model(xgb.best_estimator_, X_test, y_test),
    "Best CatBoost": eval_model(cat.best_estimator_, X_test, y_test),
}

# Посмотрим лучшие параметры
print("Best Params RF:", rf.best_params_)
print("Best Params XGB:", xgb.best_params_)
print("Best Params Cat:", cat.best_params_)

# Посмотрим метрики
print("\n Metrics")
print(pd.DataFrame(results).T)

Fitting 3 folds for each of 81 candidates, totalling 243 fits
Fitting 3 folds for each of 432 candidates, totalling 1296 fits
Fitting 3 folds for each of 54 candidates, totalling 162 fits
Best Params RF: {'max_depth': 20, 'min_samples_leaf': 5, 'min_samples_split': 2, 'n_estimators': 150}
Best Params XGB: {'booster': 'gbtree', 'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 4, 'n_estimators': 100, 'reg_alpha': 1, 'reg_lambda': 5, 'subsample': 0.8, 'tree_method': 'gpu_hist'}
Best Params Cat: {'depth': 6, 'iterations': 100, 'l2_leaf_reg': 5, 'learning_rate': 0.05}

 Metrics
                          MAE        RMSE        R2
Best RandomForest  107.846928  160.963393  0.128087
Best XGBoost       106.658306  161.657343  0.120553
Best CatBoost      111.840623  161.209746  0.125416


#### Оценим полученные метрики в контексте статистик целевых переменных.

In [50]:
# Целевые переменные
target = ['IC50, mM', 'CC50, mM', 'SI']

# Описательная статистика
target_stats = edata[target].describe().T
target_stats['median'] = edata[target].median()

print(target_stats[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max', 'median']])

          count        mean         std       min        25%         50%  \
IC50, mM  734.0  115.896009  156.219852  0.108830  15.099820   43.049903   
CC50, mM  734.0  384.109715  367.306455  0.700808  79.059629  258.410909   
SI        734.0    7.674797    8.947304  0.011489   1.787829    3.545013   

                 75%          max      median  
IC50, mM  138.147153   705.293226   43.049903  
CC50, mM  623.450022  1536.043255  258.410909  
SI          9.893714    38.168094    3.545013  


Выводы:
- RandomForest показывает лучшие метрики после настройки гиперпараметров.
- Ошибка предсказания (MAE) почти равна среднему значению самой переменной.
- Модель делает грубые предсказания, близкие по точности к простой константе.
- Польза модели ограничена — она пока не захватывает сложные закономерности в данных.
- R2 подтверждает это: модель объясняет очень небольшую часть дисперсии IC50

Рекомендации:
- Сделать ансамбль из лучших моделей: RandomForest, XGBoost.
- Применить PCA дифференцированно по группам признаков, чтобы учесть специфику различных блоков данных и избежать потерь важной информации при общей компрессии.
- Обратить внимание на биологическую значимость признаков